In [1]:
!pip install -q vllm outlines pydantic qwen-vl-utils transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50

In [2]:
%%writefile vllm_engine.py
import os
import json
import glob
import time
import re
import torch
import multiprocessing as mp
import queue
import base64

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
INPUT_PATTERN = "/kaggle/input/datasets/bumbleboo/aic26-b2-taylor/dataset/Keyframes_*/*"
OUTPUT_DIR = "/kaggle/working/metadata"

PROMPT_TEXT = (
    "1. Extract all text (OCR) present in the image, with special attention to accurate Vietnamese diacritics. If there is no text, return an empty string \"\". Do NOT hallucinate or repeat characters.\n"
    "2. Provide a short description of the image context. You MUST write this caption STRICTLY IN ENGLISH.\n"
    "Your response must be ONLY a valid JSON object with exactly two keys: 'ocr_text' and 'caption'. Do not include any markdown formatting, explanations, or extra text."
)

def extract_json_from_text(text):
    try:
        clean_text = text.strip()
        if clean_text.startswith("```json"):
            clean_text = clean_text[7:]
        if clean_text.endswith("```"):
            clean_text = clean_text[:-3]
        
        match = re.search(r'\{.*\}', clean_text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
            
        return {"caption": "ERROR_NO_JSON", "ocr_text": text.strip()[:100]}
    except Exception:
        return {"caption": "ERROR_PARSE", "ocr_text": text.strip()[:100]}

def encode_image_base64(image_path):
    """Đọc ảnh và mã hóa sang Base64 để tránh lỗi parse URL"""
    with open(image_path, "rb") as image_file:
        b64_string = base64.b64encode(image_file.read()).decode('utf-8')
    return f"data:image/jpeg;base64,{b64_string}"

def vllm_worker(task_queue, gpu_id, output_dir, model_id):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    os.environ["USE_TF"] = "0"
    os.environ["USE_JAX"] = "0"
    
    from vllm import LLM, SamplingParams
    
    print(f"🚀 [GPU {gpu_id}] Đang khởi tạo vLLM...", flush=True)
    llm = LLM(
        model=model_id,
        tensor_parallel_size=1,
        dtype="half",
        gpu_memory_utilization=0.92,
        max_model_len=2048,
        enforce_eager=True, 
        limit_mm_per_prompt={"image": 1},
        mm_processor_kwargs={"max_pixels": 768 * 768, "min_pixels": 256 * 256},
    )

    sampling_params = SamplingParams(
        temperature=0.1,
        max_tokens=256,
        repetition_penalty=1.1, 
        stop=["```", "}\n"]
    )

    while True:
        try:
            video_dir = task_queue.get(timeout=15)
        except queue.Empty:
            break

        if video_dir is None:
            print(f"🛑 [GPU {gpu_id}] Tín hiệu dừng. Queue trống.", flush=True)
            break

        video_id = os.path.basename(video_dir)
        output_file = os.path.join(output_dir, f"{video_id}.json")

        if os.path.exists(output_file):
            print(f"[GPU {gpu_id}] Đã có {video_id}.json, bỏ qua...", flush=True)
            continue

        image_files = sorted(glob.glob(os.path.join(video_dir, "*.jpg")))
        if not image_files:
            continue

        print(f"🎥 [GPU {gpu_id}] Đang xử lý: {video_id} ({len(image_files)} frames)", flush=True)

        # TRUYỀN ẢNH DƯỚI DẠNG CHUỖI BASE64 ĐỂ FIX LỖI URL
        messages = []
        for img in image_files:
            b64_url = encode_image_base64(img)
            messages.append([
                {
                    "role": "user", 
                    "content": [
                        {"type": "image_url", "image_url": {"url": b64_url}}, 
                        {"type": "text", "text": PROMPT_TEXT}
                    ]
                }
            ])

        try:
            outputs = llm.chat(messages=messages, sampling_params=sampling_params, use_tqdm=True)
        except Exception as e:
            print(f"❌ [GPU {gpu_id}] LỖI infer {video_id}: {e}", flush=True)
            continue

        frames_data = []
        for img_path, out in zip(image_files, outputs):
            raw_text = out.outputs[0].text
            result_dict = extract_json_from_text(raw_text)

            frames_data.append({
                "frame_id": os.path.basename(img_path),
                "caption": result_dict.get("caption", ""),
                "ocr_text": result_dict.get("ocr_text", ""),
            })

        final_json_object = {
            "video_id": video_id,
            "n_keyframes": len(image_files),
            "frames": frames_data,
        }

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(final_json_object, f, ensure_ascii=False, indent=2)

        print(f"✅ [GPU {gpu_id}] Đã lưu: {video_id}.json", flush=True)

    print(f"🛑 [GPU {gpu_id}] Đã hết việc!", flush=True)

def main():
    mp.set_start_method('spawn', force=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    all_video_dirs = sorted(glob.glob(INPUT_PATTERN))
    valid_dirs = [d for d in all_video_dirs if os.path.isdir(d)]
    num_gpus = torch.cuda.device_count() or 1

    print(f"✅ Nạp {len(valid_dirs)} thư mục video vào Queue.", flush=True)
    task_queue = mp.Queue()
    for video_dir in valid_dirs: task_queue.put(video_dir)
    for _ in range(num_gpus): task_queue.put(None)

    processes = []
    for gpu_id in range(num_gpus):
        p = mp.Process(target=vllm_worker, args=(task_queue, gpu_id, OUTPUT_DIR, MODEL_ID))
        p.start()
        processes.append(p)
        if gpu_id < num_gpus - 1: time.sleep(15)

    for p in processes: p.join()
    print(f"🎉 HOÀN TẤT!", flush=True)

if __name__ == '__main__': main()

Writing vllm_engine.py


In [3]:
!python vllm_engine.py

✅ Nạp 2 thư mục video vào Queue.
🚀 [GPU 0] Đang khởi tạo vLLM...
INFO 08-25 16:13:03 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 1}, 'mm_processor_kwargs': {'max_pixels': 589824, 'min_pixels': 65536}, 'model': 'Qwen/Qwen3-VL-2B-Instruct'}
config.json: 1.50kB [00:00, 3.03MB/s]
🚀 [GPU 1] Đang khởi tạo vLLM...
INFO 08-25 16:13:09 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 1}, 'mm_processor_kwargs': {'max_pixels': 589824, 'min_pixels': 65536}, 'model': 'Qwen/Qwen3-VL-2B-Instruct'}
INFO 08-25 16:13:21 [model.py:645] Resolved architecture: Qwen3VLForConditionalGeneration
WARNING 08-25 16:13:21 [model.py:2217] Casting torch.bfloat16 to torch.float16.
INFO 08-25 16:13:21 [model.py:1883] Using max model len 2048
preprocessor_config.json: 100%|████████████████| 390